Aquí tienes una propuesta metodológica y estructural para un **Diseño de Clase Universitaria o Técnica**. Está pensada para una sesión de **2 a 3 horas prácticas** bajo la estrategia de codificación en vivo (*live coding*) y aprendizaje acumulativo utilizando el cuaderno de Jupyter y SQLite.

---

# 📋 DISEÑO DE CLASE: DOMINIO PROGRESSIVO DE CONSULTAS EN SQL

**Asignatura:** Fundamentos de Bases de Datos / Analítica de Datos

**Unidad:** Extracción y Filtrado de Estructuras Relacionales

**Base de Datos de Trabajo:** `monitoreo_y_gestion.db`

---

## 🎯 Objetivos de Aprendizaje

Al finalizar la sesión, el estudiante estará en la capacidad de:

1. Construir consultas de extracción de datos respetando la jerarquía lógica del motor SQL.
2. Filtrar registros combinando operadores relacionales y lógicos dentro de la cláusula `WHERE`.
3. Segmentar información mediante agregaciones estadísticas con `GROUP BY`.
4. Controlar el rendimiento de la salida utilizando ordenamientos estructurados y límites de filas.

---

## 🗺️ Mapa del Viaje Lógico de SQL (Sintaxis vs. Ejecución)

> ⚠️ **El gran choque conceptual:** Uno de los errores más comunes en los estudiantes es creer que SQL lee el código en el mismo orden en que se escribe. Para evitar frustraciones, inicia la clase mostrando esta diferencia fundamental:

| Orden de Escritura (Sintaxis) | Orden de Procesamiento Interno (Motor SQL) |
| --- | --- |
| `1. SELECT` (¿Qué columnas quiero ver?) | `1. FROM` (¿Dónde están los datos?) |
| `2. FROM` (¿De qué tabla?) | `2. WHERE` (¿Qué filas descarto primero?) |
| `3. WHERE` (¿Bajo qué condiciones?) | `3. GROUP BY` (¿Cómo agrupo el remanente?) |
| `4. GROUP BY` (¿Bajo qué criterio agrupo?) | `4. SELECT` (Cálculo de funciones y proyección) |
| `5. ORDER BY` (¿Cómo lo ordeno?) | `5. ORDER BY` (Ordenamiento final de la salida) |
| `6. LIMIT` (¿Cuántos registros muestro?) | `6. LIMIT` (Corte de la matriz resultante) |

---

## ⏱️ Cronograma y Desarrollo de la Clase

### 🟢 Bloque 1: Introducción e Inspección del Entorno (15 min)

* **Actividad del Docente:** Conectar el cuaderno de Jupyter a la base de datos y mapear las tablas disponibles. Explicar que utilizaremos la tabla `desempeño_rrhh` como laboratorio conceptual.
* **Código Base de Conexión:**

```python
import sqlite3
import pandas as pd

conexion = sqlite3.connect("monitoreo_y_gestion.db")
# Comprobación de tablas activas
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conexion)

```

---

### 🔵 Bloque 2: La Proyección Lineal (`SELECT` + `FROM`) (25 min)

* **Concepto:** Extraer columnas específicas de una matriz relacional.
* **Analogía Visual:** "Es un corte vertical a la tabla". Solo modificamos el ancho del reporte, no el largo.
* **Ejercicio Guiado:** Listar los empleados con su departamento y su sueldo.

```sql
SELECT ID_Empleado, Departamento, Salario_Mensual_USD 
FROM desempeño_rrhh;

```

* **Reto Rápido para el Alumno:** Modificar la consulta para traer únicamente el identificador y la puntuación de evaluación.

---

### 🟡 Bloque 3: El Embudo Horizontal (`WHERE`) (30 min)

* **Concepto:** Filtrado condicional de filas.
* **Analogía Visual:** "Es un corte horizontal". El motor evalúa cada registro y descarta los que devuelven `FALSE`.
* **Caso Práctico:** Buscar perfiles en modalidad 'Remoto' que ganen más de $6,000 USD.

```sql
SELECT ID_Empleado, Modalidad, Salario_Mensual_USD 
FROM desempeño_rrhh
WHERE Modalidad = 'Remoto' AND Salario_Mensual_USD > 6000.0;

```

* **Punto de Discusión en Clase:** ¿Qué pasa si cambiamos el `AND` por un `OR`? Analizar cómo se expande o contrae el universo de datos devuelto.

---

### 🟠 Bloque 4: El Colapso de Filas y Agregación (`GROUP BY`) (40 min)

* **Concepto:** Reducir múltiples registros individuales en un único valor de resumen por categoría.
* **Explicación Crítica:** No puedes colocar una columna común al lado de una función de agregación (`AVG`, `SUM`, `COUNT`) a menos que esa columna esté declarada en el `GROUP BY`.
* **Caso Práctico:** Calcular el salario promedio y la evaluación más alta por cada Departamento.

```sql
SELECT 
    Departamento, 
    AVG(Salario_Mensual_USD) AS Salario_Promedio,
    MAX(Puntuacion_Evaluacion) AS Evaluacion_Maxima
FROM desempeño_rrhh
GROUP BY Departamento;

```

---

### 🔴 Bloque 5: Clasificación y Control de Flujo (`ORDER BY` + `LIMIT`) (30 min)

* **Concepto:** Crear rankings de datos basados en métricas previamente calculadas.
* **Caso Práctico:** Encontrar el departamento con el peor rendimiento promedio del personal que trabaja de forma presencial.

```sql
SELECT 
    Departamento, 
    AVG(Puntuacion_Evaluacion) AS Rendimiento_Promedio
FROM desempeño_rrhh
WHERE Modalidad = 'Presencial'
GROUP BY Departamento
ORDER BY Rendimiento_Promedio ASC
LIMIT 1;

```

* **Análisis del Flujo:** Rastrear paso a paso con los estudiantes por qué el `LIMIT 1` debe ir obligatoriamente al final para evitar que el sistema trunque los datos antes de computar los promedios reales de las áreas.

---

## Evaluación Formativa (Cierre de Clase)

Para validar que los conceptos quedaron claros, dicta el siguiente reto de código independiente a los alumnos en los últimos 15 minutos:

> **Reto de Cierre:** "Utilizando la tabla `contaminacion_ambiental`, escribe una consulta SQL que devuelva la Ciudad que registró la temperatura promedio más alta, evaluando únicamente los días en los que la calidad del aire se clasificó como 'Buena'. Utiliza alias legibles para tus cálculos."

**Solución Esperada del Estudiante:**

```sql
SELECT Ciudad, AVG(Temperatura_C) AS Temp_Promedio
FROM contaminacion_ambiental
WHERE Clasificacion_Calidad = 'Buena'
GROUP BY Ciudad
ORDER BY Temp_Promedio DESC
LIMIT 1;

```

---

## Consejos Didácticos para el Profesor

* **Visualiza los errores:** Permite intencionalmente que los alumnos escriban `WHERE AVG(Salario_Mensual_USD) > 5000`. Deja que el motor arroje el error y úsalo para explicar que el `WHERE` no puede filtrar agregaciones porque se ejecuta *antes* de que los grupos existan en la memoria.
* **Pandas como puente:** Al usar `pd.read_sql_query()`, los estudiantes pueden ver la salida inmediatamente como un DataFrame limpio y estético, lo cual incrementa la motivación y conecta las bases de datos con el ecosistema de la Ciencia de Datos.